In [ ]:
!pip install polars

In [ ]:
import yaml
import numpy as np
import polars as pl
import polars.selectors as cs
from tqdm import tqdm
import statsmodels.api as sm
from scipy.stats import spearmanr
import functools, operator
from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

## Define analysis params

In [ ]:
mac = 200
variant_class = 'all_variants'
# ancestries = ['EUR', 'AFR', 'AMR']
ancestries = ['META']

selected_categories = ['plof'] # missense

only_snps = False  # Whether to include only SNPs (exclude indels)%%!
only_clinvar = False
exclude_clinvar = False  # Independent toggle: exclude ClinVar-annotated variants


BUCKET_DIR = "/home/jupyter/workspace/aou-gym-processed-data"
DATA_DIR = f"{BUCKET_DIR}/allxall_exome_sumstats"
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'utils' / 'variant_filtering.py').exists())
CONFIG_DIR = str(REPO_ROOT / "configs" / "all_x_all")

variant_class_path = f"{CONFIG_DIR}/config_variant_classes.yaml"
with open(variant_class_path) as f:
    variant_class_config = yaml.safe_load(f)
    
vc = variant_class_config[variant_class]
vc_filters = vc['variant_filtering']

def compile_filter(node):
    """Turn a YAML filter node into a single pl.Expr."""
    if isinstance(node, str):
        return eval(node)                       # leaf: a Polars expression string
    if isinstance(node, list):
        # bare list defaults to OR (matches the 'coding' semantics)
        return functools.reduce(operator.or_, (compile_filter(n) for n in node))
    if isinstance(node, dict):
        if 'any_of' in node:
            return functools.reduce(operator.or_,  (compile_filter(n) for n in node['any_of']))
        if 'all_of' in node:
            return functools.reduce(operator.and_, (compile_filter(n) for n in node['all_of']))
        raise ValueError(f"filter dict must have 'any_of' or 'all_of', got {list(node)}")
    raise TypeError(f"unexpected filter node: {type(node)}")

print(f"Variant class: {variant_class}")
print(f"  Exclude ClinVar: {exclude_clinvar}")

# Load annotation configuration
config_path = f"{CONFIG_DIR}/config_correlations.yaml" 

with open(config_path) as f:
    config = yaml.safe_load(f)
    
records = [
    {
        "category": category,
        "annotation": anno,
        "color": props["color"],
        "label": props["label"],
        "annotation_dir": props.get("direction", 1),
    }
    for category, annos in config["rare_variant_annotations"].items()
    for anno, props in annos.items()
]

# Create the DataFrame directly from the list of records
anno_config_df = (
    pl.DataFrame(records)
    .filter(
        pl.col("category").is_in(selected_categories)
    )
    .with_columns(
        pl.col("annotation_dir").cast(pl.Int8)
    )
)

all_annotation_list = anno_config_df.select(pl.col("annotation")).to_series().to_list()

anno_config_df

## Process gene-trait associations

In [ ]:
RVAT_DIR = f"{BUCKET_DIR}/rvat_gene_results"
ASSOC_FILE = "SAIGE_pLoF_sig_META_MAF0001"
gene_trait_df = pl.read_parquet(f"{RVAT_DIR}/{ASSOC_FILE}.parquet")

# FDR needs no nulls in the p-value array
gene_trait_df = (
    gene_trait_df
    .with_columns(
        region = pl.col('gene_id'),
        phenotype = pl.col('phenoname')
    )
)

gene_trait_df

## Process annotations

In [ ]:
# anno = pl.scan_parquet("/home/jupyter/workspace/processed_data/ukbgym_exome_sumstats/variant_annotations_exome_all_genes.parquet")
# anno = pl.scan_parquet(f"{DATA_DIR}/variant_metadata_exome_gym_genes_annotated.parquet")
anno = pl.scan_parquet(f"{DATA_DIR}/aou_exome_variant_qc_GYM_annotated_subset.parquet")

# vc_filters is now the 'variant_filtering' node (list or dict), not a flat list of strings
if vc_filters is not None:
    variant_expr = compile_filter(vc_filters)
else:
    variant_expr = pl.lit(True)

# clinvar / snp filters are still simple ANDs on top
extra = []
if exclude_clinvar:
    extra.append(pl.col('clinical_significance').is_null())
elif only_clinvar:
    extra.append(pl.col('clinical_significance').is_not_null())
if only_snps:
    extra.append((pl.col('ref').str.len_chars()==1) & (pl.col('alt').str.len_chars()==1))
    
anno = (
    anno
    .with_columns(
        is_ins = pl.col('ref').str.len_chars() < pl.col('alt').str.len_chars(),
        is_del = pl.col('ref').str.len_chars() > pl.col('alt').str.len_chars(),
    )
    .filter(
        # Always-applied filters
        (pl.col('region').is_in(gene_trait_df['region'].unique())),
        (pl.col('variant_length') <= 50),

        # Dynamic filters from variant_class.yaml + exclude_clinvar
        variant_expr,        # the OR/AND tree, as one expression
        *extra,              # these AND with everything
    )
    .with_columns(
        inframe_deletion = (pl.col('variant_length') % 3 == 1) & (pl.col('is_del')==True), #SNPs have length 1
        inframe_insertion = (pl.col('variant_length') % 3 == 1) & (pl.col('is_ins')==True),

        clinvar_patho = pl.col('clinical_significance').str.contains('(?i)Pathogenic').fill_null(False),
        clinvar_likely_patho = pl.col('clinical_significance').str.contains('(?i)Likely_pathogenic').fill_null(False),
        clinvar_benign = pl.col('clinical_significance').str.contains('(?i)Benign').fill_null(False),
        clinvar_likely_benign = pl.col('clinical_significance').str.contains('(?i)Likely_benign').fill_null(False),
    )
)

selected_annos = anno_config_df.filter(
    pl.col('category').is_in(selected_categories)
)['annotation'].to_list()

existing_annos = [c for c in all_annotation_list if c in anno.collect_schema().names()]
selected_annos = list(set(selected_annos).intersection(set(existing_annos)))
fillna_cols = [c+'_is_na' for c in selected_annos]

anno = (
    anno
    .select(
        set(['id', 'region']).union(set(selected_annos))
    )
    .collect(engine='streaming')
    # .drop_nulls()
)

anno

In [ ]:
melted_anno = (
    anno.lazy()

    .select(
        set(['id', 'region']).union(set(selected_annos))
    )

    .unpivot(
        index=["id", "region"],
        on=selected_annos,
        variable_name="annotation",
        value_name="annotation_score"
    )
    .with_columns(
        pl.col("annotation_score").cast(pl.Float32),
        pl.col("region").cast(pl.Utf8),
    )
        
    # Keep variants that don't have fillna annotation
    # .join(
    #     anno_fillna_melted,
    #     on=['id', 'region', 'annotation'],
    #     how='semi'
    # )
    .drop_nulls()
    
    .collect(engine='streaming')
)

melted_anno

## Process phenotypes (appv file)

In [ ]:
# pheno_appv = pl.scan_parquet(f"{DATA_DIR}/appv_exome_sumstats_associations_all_ancestries.parquet")
appv_list = [
    'appv_physical_measurement.parquet',
    'appv_lab_measurement.parquet',
    'appv_mcc2_phecodex.parquet',
    'appv_r_drug.parquet'
]

pheno_appv = (
    pl.concat(
        [pl.scan_parquet(f"{DATA_DIR}/variant_sumstats_associations_META/{appv_file}") for appv_file in appv_list]
    )
    .select(['id', 'phenotype', 'AC', 'BETA', 'SE'])
)


# Filter to variants in annotation set and low MAC
anno_keys = melted_anno.select(pl.col('id').unique()).lazy()

# Merge phenotype data and annotation data
appv = (
    pheno_appv
    .join(
        anno_keys, on='id', how='semi'
    )
    .filter(
        pl.col('AC') <= mac,
    )
)

appv.head().collect()

## Join and get stats

In [ ]:
id_region = anno.select(['id', 'region']).unique().lazy()

# Build the main lazy query plan
# Join order: appv → id_region (adds region) → gene_trait_df (filter early) → melted_anno (annotation scores)
final_lazy_plan = (
    appv
    .join(id_region, on='id', how='inner')
    .join(
        gene_trait_df[["region", "phenotype"]].lazy(),
        on=["region", "phenotype"],
        how="inner"
    )
    .join(
        melted_anno.lazy(),
        on=["id", "region"],
        how="inner"
    )

    # Spearman: rank with "average" for proper tie handling
    .with_columns(
        pl.col(c)
        .rank("average")
        .over(["region", "phenotype", "annotation"])
        .alias(f"{c}_rank")
        for c in ['annotation_score', 'BETA']
    )

    .group_by(["region", "phenotype", "annotation"])
    .agg(
        n_variants = pl.col("id").count(),
        loftee_corr = pl.when(
            (pl.col("annotation_score_rank").n_unique() > 1) & 
            (pl.col("BETA_rank").n_unique() > 1)
        )
        .then(
            pl.corr("annotation_score_rank", "BETA_rank", propagate_nans=True)
        )
        .otherwise(None)
    )
    
    .drop_nans().drop_nulls()
    
    .collect(engine='streaming')
)

# Join with beta directions and annotation directions
correlation_df = (
    final_lazy_plan
    
    .join(
        gene_trait_df.select(['region', 'phenotype', 'category', 'total_variants', 'trait_type', 'n_cases', 'n_controls', 'description', 'disease_category', 'ancestries', 'META_MAC', 'META_Pvalue_SKATO', 'META_Pvalue_Burden']),
        on=['region', 'phenotype']
    )
)

print("Final DataFrame shape:", correlation_df.shape)
correlation_df

In [ ]:
n_per_cat = (
    correlation_df
    .group_by('category')
    .agg(pl.len().alias('n'))
    .with_columns(pl.format('n={}', pl.col('n')).alias('label'))
)

(
    ggplot(correlation_df, aes(x='category', y='loftee_corr'))
    + geom_boxplot()
    + geom_text(
        data=n_per_cat,
        mapping=aes(x='category', label='label'),
        y=correlation_df['loftee_corr'].min(),
        va='top',
        size=9,
        inherit_aes=False,
    )
    + labs(title='Distribution of Correlations', x='Category', y='Correlation')
    + theme_minimal()
    + theme(
        panel_grid_major=element_line(color='lightgray', size=0.5),
        panel_grid_minor=element_line(color='lightgray', size=0.25),
        plot_background=element_rect(fill="white", color="white"),
    )
)

In [ ]:
correlation_df.write_parquet(f"{RVAT_DIR}/{ASSOC_FILE}_loftee_correlations.parquet")